In [1]:
import sys


In [2]:
import os
import sys
import json
import ast
import re
import nest_asyncio, asyncio
import logging
from pathlib import Path
import pyarrow as pa

from datasets import Dataset, Features, Sequence, Value, load_from_disk, concatenate_datasets
from transformers import AutoTokenizer
from google import genai
from google.genai import types

In [3]:
from importlib import reload

import entity_discovery_prompts as prompts_module
reload(prompts_module)

from entity_discovery_prompts import (
    SYSTEM_CONTEXT,
    example_user_1, example_assistant_1, example_explanation_1,
    example_user_2, example_assistant_2, example_explanation_2,
    example_user_3, example_assistant_3, example_explanation_3,
    example_user_negative_1, example_assistant_negative_1, example_explanation_negative_1,
)


In [4]:
GEMINI_API_KEY  = os.getenv("GEMINI_API_KEY")
if not GEMINI_API_KEY:
    raise RuntimeError("GEMINI_API_KEY env var not set – please `export GEMINI_API_KEY=…` before launching.")
    
MODEL = "gemini-2.0-flash"
model_name = "gemini-2.0"

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

SAVE_CHUNK      = 1000               # save result every SAVE_CHUNK examples
PRINTOUT        = True
TAKE_SUBSET     = False
SUBSET_SIZE     = 5
BATCH_SIZE      = 128               # how many concurrent Gemini calls
TIME_SLEEP = 0.05

TTL = '259200s' # for cache
PRINT_RAW_RESPONSE = False


In [5]:
logging.basicConfig(level=logging.INFO, force=True)

logger = logging.getLogger(__name__)
logger.setLevel(logging.INFO)

handler = logging.StreamHandler(sys.stdout)
handler.setLevel(logging.INFO)
fmt = logging.Formatter("%(asctime)s %(levelname)s %(message)s")
handler.setFormatter(fmt)
logger.addHandler(handler)

# Quiet down other noisy loggers
logging.getLogger("google_genai.models").setLevel(logging.WARNING)
logging.getLogger("httpx").setLevel(logging.WARNING)

In [6]:
few_shot_examples = [
    {"user_query": example_user_1, "model_answer": example_assistant_1, "model_explanation": example_explanation_1},
    {"user_query": example_user_2, "model_answer": example_assistant_2, "model_explanation": example_explanation_2},
    {"user_query": example_user_3, "model_answer": example_assistant_3, "model_explanation": example_explanation_3},
    # negative examples
    {"user_query": example_user_negative_1, "model_answer": example_assistant_negative_1, "model_explanation_negative": example_explanation_negative_1},
]

STRUCTURED_CONTENT_FEW_SHOTS = [
    {"role": "user", "parts": [{"text": "I will provide you with examples"}]},
    {"role": "model", "parts": [{"text": "Understood — send the sample and I'll output entities"}]},
]

for ex in few_shot_examples:
    STRUCTURED_CONTENT_FEW_SHOTS.append({
        "role": "user",
        "parts": [{"text": ex["user_query"]}]
    })
    STRUCTURED_CONTENT_FEW_SHOTS.append({
        "role": "model",
        "parts": [{"text": ex["model_answer"]}]
    })
    if "model_explanation" in ex:
        STRUCTURED_CONTENT_FEW_SHOTS.append({
            "role": "user",
            "parts": [{"text": "Explanation of the previous output:"}]
        })
        STRUCTURED_CONTENT_FEW_SHOTS.append({
            "role": "model",
            "parts": [{"text": ex["model_explanation"]}]
        })
    if "model_explanation_negative" in ex:
        STRUCTURED_CONTENT_FEW_SHOTS.append({
            "role": "user",
            "parts": [{"text": "Explanation of what is wrong with the previous output:"}]
        })
        STRUCTURED_CONTENT_FEW_SHOTS.append({
            "role": "model",
            "parts": [{"text": ex["model_explanation_negative"]}]
        })

STRUCTURED_CONTENT_FEW_SHOTS.append({
    "role": "user",
    "parts": [{"text": "**End of examples**.\nNow read the actual input:"}]
})


In [7]:
def extract_rightmost_list(response: str) -> list:
    # 1) Strip any lines that start with ``` (with or without language tag)
    #    but kepp what's between them.
    response = re.sub(r'(?m)^```.*\n?', "", response)

    # 2) Find all [ ... ] spans (non-greedy, across newlines)
    matches = re.findall(r"\[.*?\]", response, flags=re.DOTALL)
    if not matches:
        return []
    candidate = matches[-1]

    # 3) Try JSON, then Python literal
    try:
        return json.loads(candidate)
    except json.JSONDecodeError:
        pass
    try:
        return ast.literal_eval(candidate)
    except Exception:
        return []

In [ ]:
async def gemini_call(client, user_query: str, count_only, cache=None, cache_system_prompt=True, max_retries: int=3) -> str:
    global input_tokens, output_tokens
    call_fn = client.aio.models.count_tokens if count_only else client.aio.models.generate_content

    if cache is not None:
        config = types.GenerateContentConfig(cached_content=cache.name)
    elif not cache_system_prompt:
        config = types.GenerateContentConfig(system_instruction=SYSTEM_CONTEXT)
    else:
        config = None

    if cache_system_prompt:
        contents = [{"role": "user", "parts": [{"text": user_query}]}]
    else:
        contents = []
        contents.extend(STRUCTURED_CONTENT_FEW_SHOTS)
        contents.append({"role": "user", "parts": [{"text": user_query}]})
        
    for attempt in range(1, max_retries + 1):
        try:            
            resp = await call_fn(model=MODEL, contents=contents, config=config)
            if count_only:
                input_tokens += resp.total_tokens
                return ""
            metadata = resp.usage_metadata
            pt = metadata.prompt_token_count or 0
            cc = metadata.cached_content_token_count or 0
            ct = metadata.candidates_token_count or 0
            input_tokens += pt - cc
            output_tokens += ct
            return getattr(resp, "text", "") or ""
        except Exception as e:
            logger.warning(f"Gemini call attempt {attempt}/{max_retries} failed: {e!r}")
            if attempt < max_retries:
                await asyncio.sleep(0.5 * attempt)
            else:
                logger.error(f"All {max_retries} Gemini attempts failed for query: {user_query[:50]!r}")
                return ""


In [9]:
async def main(dataset_dir: str, path_to_save, count_only=False, cache_system_prompt=True, start_idx: int = 0, end_idx=None):
    ds_full = Dataset.load_from_disk(str(dataset_dir))
    if TAKE_SUBSET:
        overall_end = min(start_idx + SUBSET_SIZE, len(ds_full))
    else:
        overall_end = len(ds_full) if end_idx is None else min(len(ds_full), end_idx)

    client = genai.Client(api_key=GEMINI_API_KEY)
    cache = None

    try:
        if cache_system_prompt:
            cache = await create_cache(client, count_only)
            
        for chunk_start in range(start_idx, overall_end, SAVE_CHUNK):
            chunk_end = min(chunk_start + SAVE_CHUNK, overall_end)
            ds_chunk = ds_full.select(range(chunk_start, chunk_end))
            entities = await run_all(client, ds_chunk, chunk_start, chunk_end, cache, count_only=count_only)
            
            assert len(entities) == ds_chunk.num_rows, f"{len(entities)=} vs {ds_chunk.num_rows=}"
            if not count_only:
                # coerce any stray string into a single‐element list
                fixed = [[str(x) for x in t] if isinstance(t, list) else [str(t)] for t in entities]
                entities_arr = pa.array(fixed, type=pa.list_(pa.string()))
                ds_chunk = ds_chunk.add_column("response", fixed)
                if TAKE_SUBSET:
                    save_name = f"{model_name}_subset_{SUBSET_SIZE}_{chunk_start}-{chunk_end}"
                else:
                    save_name = f"{model_name}_{chunk_start}-{chunk_end}"
        
                out_dir = os.path.join(str(path_to_save), save_name)
                ds_chunk.save_to_disk(out_dir)
                logger.info(f"Saved chunk {chunk_start}-{chunk_end} to {out_dir}")
    finally:
        if cache is not None:
            await client.aio.caches.delete(name=cache.name)

In [14]:
input_tokens = 0
output_tokens = 0
cached_tokens = 0

count_only = False  # set True for free token count only
cache_system_prompt = False  # caching requires >4096 tokens; disabled for small prompts

dataset_dir = Path('../../json_data/tokenized/ch1_text_only_no_equations_2b0646f4_tokenized')
path_to_save = Path('../../json_data/entity_discovery_output')
logger.info(f'request with {model_name}')
await main(dataset_dir, path_to_save, count_only=count_only, cache_system_prompt=False, start_idx=0, end_idx=None)

print('===finished===')
print('input_tokens:', input_tokens)
print('output_tokens:', output_tokens)
print('cached_tokens', cached_tokens)


2026-05-27 17:07:27,272 INFO request with gemini-2.0


INFO:__main__:request with gemini-2.0


2026-05-27 17:07:34,346 WARNING Gemini call attempt 1/3 failed: ClientError("429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \\n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.0-flash\\n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.0-flash\\n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_input_token_count, limit: 0, model: gemini-2.0-flash\\nPlease retry in 25.582654517s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https

2026-05-27 17:07:34,397 WARNING Gemini call attempt 1/3 failed: ClientError("429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \\n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.0-flash\\n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.0-flash\\n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_input_token_count, limit: 0, model: gemini-2.0-flash\\nPlease retry in 25.575626008s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https

2026-05-27 17:07:34,398 WARNING Gemini call attempt 1/3 failed: ClientError("429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \\n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_input_token_count, limit: 0, model: gemini-2.0-flash\\n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.0-flash\\n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.0-flash\\nPlease retry in 25.573189772s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https

...

INFO:__main__:Gemini: processed 136/136


Saving the dataset (0/1 shards):   0%|          | 0/136 [00:00<?, ? examples/s]

2026-05-27 17:07:45,288 INFO Saved chunk 0-136 to ../../json_data/entity_discovery_output/gemini-2.0_0-136


INFO:__main__:Saved chunk 0-136 to ../../json_data/entity_discovery_output/gemini-2.0_0-136


===finished===
input_tokens: 0
output_tokens: 0
cached_tokens 0


In [11]:
print(input_tokens)
print(output_tokens)
print(cached_tokens)

0
0
0


## Unite chunks

In [12]:
def unite_output(dataset_path, original_dataset_size, chunk_size, llm_name, start_idx=0):
    output_path = os.path.join(dataset_path, f'{llm_name}_all')
    end_idx = 0
    datasets = []
    while end_idx < original_dataset_size:
        end_idx = min(start_idx + chunk_size, original_dataset_size)
        path = os.path.join(dataset_path, f"{llm_name}_{start_idx}-{end_idx}")
        dataset = load_from_disk(path)
        print(f'loaded from {path}')
        datasets.append(dataset)
        start_idx += chunk_size
    united_dataset = concatenate_datasets(datasets)
    united_dataset.save_to_disk(output_path)
    print('saved to', output_path)
    return output_path


In [13]:
dataset_path = '../../json_data/entity_discovery_output'
original_dataset_size = 136  # update to full dataset size after smoke test passes
saved_to_path = unite_output(dataset_path, original_dataset_size, chunk_size=1000, llm_name=model_name, start_idx=0)


loaded from ../../json_data/entity_discovery_output/gemini-2.0_0-136


Saving the dataset (0/1 shards):   0%|          | 0/136 [00:00<?, ? examples/s]

saved to ../../json_data/entity_discovery_output/gemini-2.0_all
